# Agent Frameworks and Implementation

## Overview

We start with ToyAIKit - a simple framework for learning agentic AI, debugging agents, and interacting with them. It makes everything visible, so you can see every function call the agent makes.

With ToyAIKit, we'll:

- Take the search and add_entry tools from the previous section and use them in an agent
- Organize tools into a class with docstrings and type hints - frameworks extract tool descriptions from these automatically
- Set up an interactive chat interface
- Put agent code in a separate Python file

Then we'll go from agentic RAG to agentic search. When you search something in Google, you don't get chunks of data - you get a list of articles, and then you click on the ones that look interesting. We'll build the same thing:

- A search tool that returns snippets (relevant parts of a page)
- A fetch tool that retrieves the entire page
- The agent chooses whether it needs the full page or if snippets are enough

Finally, we'll translate everything to PydanticAI.

## Framework Selection

The main goal is to show you how agentic frameworks work. Internally, they're all pretty similar. Understanding the concepts is more important than the specific framework.

For the rest of the course, I'll mostly use PydanticAI, except in a few lessons where I'll stick to ToyAIKit because of its interactivity.



In [1]:
uv add toyaikit

Note: you may need to restart the kernel to use updated packages.


c:\Users\harun\OneDrive\Documents\GitHub\anvil-llm-lab\.venv\Scripts\python.exe: No module named uv


In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [3]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": ["query"]
    }
}

In [4]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": ["filename", "title", "description", "content"]
    }
}

In [5]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search

2) in the second interation, analyze the results from the previous search
   and perform 2 more searches

3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

## Creating the Agent
In ToyAIKit, agents are called "runners". First, we need to create a tools object that holds both our tools:

In [6]:
uv add toyaikit

Note: you may need to restart the kernel to use updated packages.


c:\Users\harun\OneDrive\Documents\GitHub\anvil-llm-lab\.venv\Scripts\python.exe: No module named uv


In [7]:
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
agent_tools.add_tool(add_entry, add_entry_tool)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results based on a query string.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'The search query to look up in the index'}},
   'required': ['query']}},
 {'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'The source filename associated with the entry'},
    'title': {'type': 'string',
     'description': 'The title of the documentation entry'},
    'description': {'type': 'string',
     'description': 'A short description summarizing the entry'},
    'content': {'type': 'string',
     'description': 'The full content of the documentation entry'}},
   'required': ['filename', 'title', 'description', 'content']}}]

In [8]:
from openai import OpenAI
from toyaikit.llm import OpenAIClient

from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

In [9]:
llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface) 

## Running the Agent
Inside the agent, we have the loop method. It's almost the same as what we implemented:



In [10]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    # chat_interface=,
    llm_client=llm_client
)

In [11]:
result = agent.loop('how can I create evidently dahsbord?')

In [12]:
print(result.last_message)

To create a dashboard effectively, you can choose between two approaches: utilizing the Python API or the user interface (UI). Below are the steps for each method.

### Using the Python API:
1. **Initialize**: Ensure you have connected to your project in the Evidently Cloud or Enterprise.
2. **Add a Tab**: You can create a new tab or add a panel to an existing tab.
   ```python
   project.dashboard.add_tab("Your Tab Name")
   ```
3. **Add Panels**: Choose the metrics you want to visualize using various types of panels (counters, line plots, pie charts, etc.).
   ```python
   from evidently.sdk.models import PanelMetric
   from evidently.sdk.panels import DashboardPanelPlot

   project.dashboard.add_panel(DashboardPanelPlot(
       title="Your Panel Title",
       size="half",
       values=[PanelMetric(legend="Row count", metric="RowCount")],
       plot_params={"plot_type": "counter", "aggregation": "sum"},
   ), tab="Your Tab Name")
   ```
4. **Configure Panel Options**: Customize si

In [13]:
result.cost.total_cost

Decimal('0.00298665')

## Adding Visibility
Right now, when we run the loop, we don't see what's happening. Let's add some visibility using callbacks:



In [14]:
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

In [15]:
chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)

In [16]:
result = agent.loop(
    'how can I create evidently dahsbard?',
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received


## Continuing the Conversation
The result object lets us continue the conversation by passing previous_messages:



In [19]:
result2 = agent.loop(
    'show me the code',
    callback=runner_callback,
    previous_messages=result.all_messages
)
# The agent realized it didn't need to search again because all the code was already in the conversation history.

# You'll find similar patterns in other frameworks like PydanticAI - separating new_messages from all_messages for convenience.

-> Response received


## Q&A Loop
For an interactive chat experience, we can use the run method:


In [20]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [1]:
# Fire her up
result = agent.run()

NameError: name 'agent' is not defined

In [ ]:
result.cost.total_cost

## Summary 
ToyAIKit implements the same agentic loop we built ourselves: - The loop method handles the tool call loop (inner loop) - The run method handles the Q&A loop (outer loop) - Callbacks provide visibility into what's happening - The framework tracks cost automatically

Now that we understand how ToyAIKit works, we're ready to explore more advanced frameworks. In the next lesson, we'll look at PydanticAI.

